<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《Build a Large Language Model From Scratch》</a>（作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a>）的配套代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 内存高效的模型 weights 加载

- 本 notebook 提供了在 GPU（或 CPU）内存有限的情况下加载较大预训练或微调模型的技巧
- 具体来说，它关注的是你使用 `torch.save(model.state_dict(), "model.pth")`（例如在第5-7章中）保存模型后，想在新的会话中加载它以继续预训练或进一步微调的场景
- 虽然示例使用的是 LLM，但本 notebook 中解释的方法是通用的，适用于加载任何 PyTorch 模型，而不仅仅是 LLM

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/memory-efficient-loading/memory-efficient-loading.webp" width="800px">

**图 5.1**：不同 weights 加载策略的内存使用对比。从基础的 `load_state_dict` 到 meta device 和 sequential 加载方法，每种策略对 GPU 内存和 CPU 内存的需求各不相同。

```
加载策略                          GPU峰值内存    CPU峰值内存
──────────────────────────────────────────────────────────────
基础方法 (model.to(device) +     ┃████████████│██████████
  load_state_dict to device)     ┃  2x 模型   │  ~1x 模型
                                  ┃            │
Sequential 加载 (CPU→逐个拷贝)   ┃██████      │██████████
                                  ┃  1x 模型   │  ~1x 模型
                                  ┃            │
Meta device + Sequential         ┃██████      │██
                                  ┃  1x 模型   │  极低
                                  ┃            │
Meta device + mmap=True          ┃██████      │██████
(推荐)                            ┃  1x 模型   │  依RAM而定
──────────────────────────────────────────────────────────────
```

| 方法 | GPU 峰值内存 | CPU 峰值内存 | 适用场景 |
|------|-------------|-------------|----------|
| 基础 `load_state_dict` | ~2x 模型大小 | ~1x 模型大小 | GPU 和 CPU 内存充足 |
| Sequential 逐个加载 | ~1x 模型大小 + 单参数 | ~1x 模型大小 | GPU 内存有限 |
| Meta device + Sequential | ~1x 模型大小 | 极低 | CPU 内存有限 |
| Meta device + mmap=True | ~1x 模型大小 | 按需加载 | CPU 内存有限（推荐） |

> **核心结论**：当内存有限时，使用 meta device 创建模型骨架，再通过 sequential 或 `mmap=True` 方式加载 weights，可以显著降低峰值内存需求。

In [1]:
from importlib.metadata import version

pkgs = [
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

torch version: 2.9.1+cu130


&nbsp;
## 1. 基准测试工具

- 首先，让我们定义一些用于追踪 VRAM（GPU 内存）的实用代码
- 稍后，我们还将介绍一个用于追踪主系统 RAM（CPU 内存）的工具
- 这些函数的用途将在后面实际应用时变得清晰

In [2]:
import gc
import time
import torch


def start_memory_tracking():
    """Initialize GPU memory tracking."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    else:
        print("This notebook is intended for CUDA GPUs but CUDA is not available.")

def print_memory_usage():
    max_gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)  # Convert bytes to GB
    print(f"Maximum GPU memory allocated: {max_gpu_memory:.1f} GB")

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(3)  # some buffer time to allow memory to clear
    torch.cuda.reset_peak_memory_stats()
    max_memory_allocated = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
    print(f"Maximum GPU memory allocated: {max_memory_allocated:.1f} GB")

&nbsp;
## 2. 模型设置

- 本代码部分设置模型本身
- 这里我们使用 "large" GPT-2 模型使事情更有趣（你可以使用 "gpt2-small (124M)" 来降低内存需求和本 notebook 的执行时间）

In [3]:
from previous_chapters import GPTModel
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch04 import GPTModel



BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-xl (1558M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

- 现在，让我们看看 GPU 内存函数的实际效果：

In [4]:
start_memory_tracking()


model = GPTModel(BASE_CONFIG)
device = torch.device("cuda")
model.to(device)

print_memory_usage()

/home/rasbt/jupyterlab/reasoning/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


Maximum GPU memory allocated: 6.4 GB


- 此外，让我们通过传入一个示例 tensor 来确保模型能正常运行

In [5]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

- 接下来，假设我们正在预训练模型并保存以备后用
- 为简化起见，我们跳过实际的预训练过程，只保存初始化的模型（但同样的概念适用）

In [6]:
# Training code would go here...

model.train()
torch.save(model.state_dict(), "model.pth")

- 最后，我们删除 Python 会话中的模型和示例 tensor，以重置 GPU 内存

In [7]:
del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


&nbsp;
## 3. 基础 weights 加载

- 现在开始有趣的部分——加载预训练模型 weights
- 让我们看看加载之前保存的模型需要多少 GPU 内存

In [8]:
# Then load pretrained weights

start_memory_tracking()

model = GPTModel(BASE_CONFIG)
model.to(device)

model.load_state_dict(
    torch.load("model.pth", map_location=device, weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

Maximum GPU memory allocated: 12.8 GB


- 注意，内存是上一个会话的 2 倍
- 这是因为在短时间内，内存中同时存在两份相同的模型：
  - 第一份通过 `model.to(device)` 存在
  - 第二份通过 `model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))` 这行代码存在；最终，加载的模型 weights 会被复制到模型中，state_dict 会被丢弃，但在短暂的时间内，主模型和加载的 state_dict 同时存在于内存中
- 接下来的部分着重解决这个问题
- 但首先，让我们测试模型并重置 GPU 内存


In [9]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


- 让我们测试另一种在实践中非常流行的常见模式：

In [10]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG)

model.load_state_dict(
    torch.load("model.pth", map_location="cpu", weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

Maximum GPU memory allocated: 6.4 GB


In [11]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


- 因此，就峰值内存而言，是先在 device 上实例化模型再使用 `map_location="device"`，还是先将 weights 加载到 CPU 内存（`map_location="cpu"`）再移动到 device，并没有区别

&nbsp;
## 4. 顺序加载 weights

- 解决上一节中指出的模型 weights 在 GPU 内存中存在两份这个问题的一种方法是顺序加载模型
- 下面我们：
  - 首先将模型加载到 GPU 内存
  - 然后将模型 weights 加载到 CPU 内存
  - 最后逐个将每个参数复制到 GPU 内存


In [ ]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG).to(device)

state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

print_memory_usage()

# Sequentially copy weights to the model's parameters
with torch.no_grad():
    for name, param in model.named_parameters():
        if name in state_dict:
            param.copy_(state_dict[name].to(device))
        else:
            print(f"Warning: {name} not found in state_dict.")

print_memory_usage()

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.7 GB


- 如上所示，内存使用量比之前低得多
- 注意内存从 6.4 GB 增加到 6.7 GB，因为最初我们只有模型在内存中，然后我们同时有模型和 1 个参数 tensor 在内存中（我们临时将参数 tensor 移动到 GPU 以便通过 `.to` 将其赋值给模型）
- 总体来说，这是一个显著的改进
- 再次让我们简要测试模型，然后为下一节重置 GPU 内存

In [ ]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input, state_dict, param
cleanup()

Maximum GPU memory allocated: 0.0 GB


&nbsp;
## 5. 在低 CPU 内存下加载模型

- 在上一节中，我们通过先将 weights（state_dict）加载到 CPU 内存，再逐个复制到模型中，从而减少了 GPU 内存使用
- 但是，如果 CPU 内存也有限怎么办？
- 本节使用 PyTorch 的所谓 "meta" device 方法，在 GPU 内存大但 CPU 内存小的机器上加载模型
- 但首先，让我们定义一个方便的函数来监控 CPU 内存

In [ ]:
import os
import psutil
from threading import Thread


def memory_usage_in_gb(func, *args, **kwargs):
    process = psutil.Process(os.getpid())

    # Measure the baseline memory usage before running the function
    baseline_mem = process.memory_info().rss / 1024 ** 3  # in GB

    # Start monitoring memory in a separate thread
    mem_usage = []
    done = False

    def monitor_memory():
        while not done:
            mem_usage.append(process.memory_info().rss / 1024 ** 3)  # Convert to GB
            time.sleep(0.1)

    t = Thread(target=monitor_memory)
    t.start()

    # Run the function
    func(*args, **kwargs)

    # Stop monitoring
    done = True
    t.join()

    peak_mem_usage_gb = max(mem_usage) - baseline_mem
    return peak_mem_usage_gb


- 首先，让我们追踪上一节中顺序 weights 加载方法的 CPU 内存使用情况

In [ ]:
def load_sequentially():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG).to(device)

    state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

    print_memory_usage()

    # Sequentially copy weights to the model's parameters
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name].to(device))
            else:
                print(f"Warning: {name} not found in state_dict.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_sequentially)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.7 GB
-> Maximum CPU memory allocated: 6.3 GB


- 现在，假设我们有一台 CPU 内存少但 GPU 内存大的机器
- 我们可以通过引入 PyTorch 的所谓 "meta" device 来权衡 CPU 内存和 GPU 内存的使用
- PyTorch 的 meta device 是一种特殊的设备类型，允许你创建 tensor 而不为其数据分配实际内存，从而有效地创建 "meta" tensor
- 这对于模型分析或架构定义等任务非常有用，在这些任务中你需要 tensor 的形状和类型而不需要内存分配的开销

In [ ]:
def load_sequentially_with_meta():
    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    state_dict = torch.load("model.pth", map_location=device, weights_only=True)

    print_memory_usage()

    # Sequentially copy weights to the model's parameters
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name])
            else:
                print(f"Warning: {name} not found in state_dict.")

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(load_sequentially_with_meta)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 12.8 GB
Maximum GPU memory allocated: 12.8 GB
-> Maximum CPU memory allocated: 1.3 GB


- 如上所示，通过在 meta device 上创建模型并直接将 weights 加载到 GPU 内存中，我们有效地减少了 CPU 内存需求
- 可能有人会问："那么顺序 weights 加载是否还有必要？它与原始方法相比如何？"
- 让我们检查简单的 PyTorch weights 加载方法作为对比（来自本 notebook 中第一个 weights 加载小节）：

In [ ]:
def baseline():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG)
    model.to(device)

    model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
    model.to(device)
    model.eval();

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(baseline)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 12.8 GB
-> Maximum CPU memory allocated: 4.4 GB


- 如上所示，不使用 meta device 的 "简单" weights 加载使用了更多内存
- 换句话说，如果你的机器 CPU 内存有限，你可以使用 meta device 方法将模型 weights 直接加载到 GPU 内存中以减少峰值 CPU 内存使用

&nbsp;
## 6. 使用 `mmap=True`（推荐）

- 作为中高级 `torch.load` 用户，你可能想知道这些方法与 PyTorch 中的 `mmap=True` 设置相比如何
- PyTorch 中的 `mmap=True` 设置启用了内存映射文件 I/O，允许 tensor 直接从磁盘存储访问数据，从而在 RAM 有限时不将整个文件加载到 RAM 中来减少内存使用
- 另请参阅 [mikaylagawarecki](https://github.com/rasbt/LLMs-from-scratch/issues/402) 的有用评论
- 乍一看，它可能看起来不如上面的顺序方法高效：

In [ ]:
def best_practices():
  with torch.device("meta"):
      model = GPTModel(BASE_CONFIG)

  model.load_state_dict(
      torch.load("model.pth", map_location=device, weights_only=True, mmap=True),
      assign=True
  )

  print_memory_usage()

peak_memory_used = memory_usage_in_gb(best_practices)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
-> Maximum CPU memory allocated: 5.9 GB


- CPU RAM 使用如此之高的原因是这台机器上有足够的 CPU RAM 可用
- 但是，如果你在 CPU RAM 有限的机器上运行，`mmap` 方法会使用更少的内存

&nbsp;
## 7. 其他方法

- 本 notebook 专注于使用 PyTorch 内置的简单方法来加载 weights
- 对于 CPU 内存有限的情况，推荐的方法是上面解释的 `mmap=True` 方法
- 或者，另一种选择是暴力方法，即分别保存和加载每个 weight tensor：

In [ ]:
model = GPTModel(BASE_CONFIG)
# Assume `model` is your trained model
state_dict = model.state_dict()

# Create a directory to store individual parameter files
os.makedirs("model_parameters", exist_ok=True)

# Save each parameter tensor separately
for name, param in state_dict.items():
    torch.save(param.cpu(), f"model_parameters/{name}.pt")

del model

In [ ]:
def load_individual_weights():

    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    print_memory_usage()
    param_dir = "model_parameters"

    with torch.no_grad():
        for name, param in model.named_parameters():
            weight_path = os.path.join(param_dir, f"{name}.pt")
            if os.path.exists(weight_path):
                param_data = torch.load(weight_path, map_location="cpu", weights_only=True)
                param.copy_(param_data)
                del param_data  # Free memory
            else:
                print(f"Warning: {name} not found in {param_dir}.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_individual_weights)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.4 GB
-> Maximum CPU memory allocated: 0.3 GB
